In [1]:
!nvidia-smi

Sat Nov 21 09:34:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.4 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5281390 sha256=d1ddefb14fe457461856e309beb8a24cad95cfec9341fc34593d1a99ba915762
  Stored in directory: /root/.cache/pip/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pycuda]


In [3]:
import random
import math
import time
import numpy as np
from multiprocessing import Pool, RawArray, cpu_count

import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

N_CITIES = 200
POP_SIZE = 400
GENERATIONS = 400
TOURNAMENT_K = 5
MUTATION_RATE = 0.1
SEED = 42
BLOCK = 10

random.seed(SEED)
CITIES = [(random.random() * 100, random.random() * 100) for _ in range(N_CITIES)]
DIST = [[math.dist(a, b) for b in CITIES] for a in CITIES]
N_BLOCKS = math.ceil((POP_SIZE - 1) / BLOCK)

_pop = None
_fit = None


def _init(pop_buf, fit_buf):
    global _pop, _fit
    _pop = np.frombuffer(pop_buf, dtype=np.int32).reshape(2, POP_SIZE, N_CITIES)
    _fit = np.frombuffer(fit_buf, dtype=np.float64).reshape(2, POP_SIZE)


def route_length(route):
    return sum(DIST[route[i]][route[i - 1]] for i in range(len(route)))


def tournament(fit):
    return min(random.sample(range(POP_SIZE), TOURNAMENT_K), key=fit.__getitem__)


def ox(p1, p2):
    n = len(p1)
    a, b = sorted(random.sample(range(n), 2))
    child = [None] * n
    child[a:b] = p1[a:b]
    used = set(p1[a:b])
    fill = [g for g in p2[b:] + p2[:b] if g not in used]
    for i, pos in enumerate(list(range(b, n)) + list(range(0, a))):
        child[pos] = fill[i]
    return child


def mutate(route):
    if random.random() < MUTATION_RATE:
        i, j = random.sample(range(len(route)), 2)
        route[i], route[j] = route[j], route[i]
    return route


def make_children(task):
    cur, gen, b0, b1 = task
    pop, fit = _pop[cur], _fit[cur].tolist()
    out_pop, out_fit = _pop[1 - cur], _fit[1 - cur]
    for b in range(b0, b1):
        random.seed(SEED * 1_000_003 + gen * 1000 + b)
        for i in range(1 + b * BLOCK, min(1 + (b + 1) * BLOCK, POP_SIZE)):
            p1 = pop[tournament(fit)].tolist()
            p2 = pop[tournament(fit)].tolist()
            child = mutate(ox(p1, p2))
            out_pop[i] = child
            out_fit[i] = route_length(child)


def ga(parallel=False, workers=None):
    workers = (workers or cpu_count()) if parallel else 1
    pop_buf = RawArray('i', 2 * POP_SIZE * N_CITIES)
    fit_buf = RawArray('d', 2 * POP_SIZE)
    _init(pop_buf, fit_buf)

    random.seed(SEED)
    for i in range(POP_SIZE):
        route = random.sample(range(N_CITIES), N_CITIES)
        _pop[0][i] = route
        _fit[0][i] = route_length(route)

    pool = Pool(workers, initializer=_init, initargs=(pop_buf, fit_buf)) if parallel else None
    per = math.ceil(N_BLOCKS / workers)
    try:
        cur = 0
        for gen in range(GENERATIONS):
            best = int(_fit[cur].argmin())
            _pop[1 - cur][0] = _pop[cur][best]
            _fit[1 - cur][0] = _fit[cur][best]

            tasks = [(cur, gen, b, min(b + per, N_BLOCKS)) for b in range(0, N_BLOCKS, per)]
            if pool:
                pool.map(make_children, tasks, chunksize=1)
            else:
                make_children(tasks[0])
            cur = 1 - cur
        return float(_fit[cur].min())
    finally:
        if pool:
            pool.close()
            pool.join()


GPU_THREADS = 128

KERNEL_SRC = r"""
#include <curand_kernel.h>

#define N     @N@
#define POP   @POP@
#define TK    @TK@
#define MUT   @MUT@f

__device__ curandState g_states[POP];

extern "C" __global__ void init_rng(unsigned long long seed) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < POP) curand_init(seed, i, 0, &g_states[i]);
}

__device__ int rnd_int(curandState *s, int n) { return (int)(curand(s) % (unsigned int)n); }
__device__ float rnd_float(curandState *s) { return curand_uniform(s); }

__device__ int tournament(const float *fit, curandState *s) {
    int best = rnd_int(s, POP);
    for (int k = 1; k < TK; k++) {
        int c = rnd_int(s, POP);
        if (fit[c] < fit[best]) best = c;
    }
    return best;
}

extern "C" __global__ void find_best(const float *fit, int *best_idx) {
    int b = 0;
    for (int i = 1; i < POP; i++) if (fit[i] < fit[b]) b = i;
    *best_idx = b;
}

extern "C" __global__ void make_children(const int *pop_in, const float *fit_in,
                              int *pop_out, float *fit_out,
                              const float *dist, const int *best_idx) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= POP) return;

    if (i == 0) {
        int b = *best_idx;
        for (int k = 0; k < N; k++) pop_out[k] = pop_in[b * N + k];
        fit_out[0] = fit_in[b];
        return;
    }

    curandState s = g_states[i];

    const int *p1 = pop_in + tournament(fit_in, &s) * N;
    const int *p2 = pop_in + tournament(fit_in, &s) * N;

    int a = rnd_int(&s, N), b = rnd_int(&s, N);
    while (b == a) b = rnd_int(&s, N);
    if (a > b) { int t = a; a = b; b = t; }

    int child[N];
    unsigned char used[N];
    for (int k = 0; k < N; k++) used[k] = 0;
    for (int k = a; k < b; k++) { child[k] = p1[k]; used[p1[k]] = 1; }

    int pos = b;
    for (int k = 0; k < N; k++) {
        int g = p2[(b + k) % N];
        if (!used[g]) {
            child[pos] = g;
            pos++;
            if (pos == N) pos = 0;
        }
    }

    if (rnd_float(&s) < MUT) {
        int x = rnd_int(&s, N), y = rnd_int(&s, N);
        while (y == x) y = rnd_int(&s, N);
        int t = child[x]; child[x] = child[y]; child[y] = t;
    }
    g_states[i] = s;
    float len = 0.0f;
    for (int k = 0; k < N; k++) {
        int from = child[k == 0 ? N - 1 : k - 1];
        len += dist[from * N + child[k]];
    }
    for (int k = 0; k < N; k++) pop_out[i * N + k] = child[k];
    fit_out[i] = len;
}
"""


def build_kernels():
    src = (KERNEL_SRC.replace("@N@", str(N_CITIES)).replace("@POP@", str(POP_SIZE))
           .replace("@TK@", str(TOURNAMENT_K)).replace("@MUT@", str(MUTATION_RATE)))
    mod = SourceModule(src, no_extern_c=True)
    return (mod.get_function("init_rng"), mod.get_function("find_best"),
            mod.get_function("make_children"))


def ga_gpu(kernels, generations=GENERATIONS):
    init_rng, find_best, make_children_gpu = kernels

    random.seed(SEED)
    pop0 = np.array([random.sample(range(N_CITIES), N_CITIES) for _ in range(POP_SIZE)], dtype=np.int32)
    fit0 = np.array([route_length(r) for r in pop0.tolist()], dtype=np.float32)
    dist = np.array(DIST, dtype=np.float32)

    d_pop = [cuda.mem_alloc(pop0.nbytes), cuda.mem_alloc(pop0.nbytes)]
    d_fit = [cuda.mem_alloc(fit0.nbytes), cuda.mem_alloc(fit0.nbytes)]
    d_dist = cuda.mem_alloc(dist.nbytes)
    d_best = cuda.mem_alloc(4)
    cuda.memcpy_htod(d_pop[0], pop0)
    cuda.memcpy_htod(d_fit[0], fit0)
    cuda.memcpy_htod(d_dist, dist)

    grid = ((POP_SIZE + GPU_THREADS - 1) // GPU_THREADS, 1)
    init_rng(np.uint64(SEED), block=(GPU_THREADS, 1, 1), grid=grid)

    cur = 0
    for gen in range(generations):
        find_best(d_fit[cur], d_best, block=(1, 1, 1), grid=(1, 1))
        make_children_gpu(d_pop[cur], d_fit[cur], d_pop[1 - cur], d_fit[1 - cur],
                          d_dist, d_best, block=(GPU_THREADS, 1, 1), grid=grid)
        cur = 1 - cur

    result = np.empty(POP_SIZE, dtype=np.float32)
    cuda.memcpy_dtoh(result, d_fit[cur])
    return float(result.min())


if __name__ == "__main__":
    kernels = build_kernels()
    ga_gpu(kernels, generations=1)

    t = time.perf_counter()
    best_seq = ga(parallel=False)
    t_seq = time.perf_counter() - t
    print(f"CPU 1 watek:      najlepsza trasa = {best_seq:.2f}, czas = {t_seq:.2f} s")

    t = time.perf_counter()
    best_par = ga(parallel=True)
    t_par = time.perf_counter() - t
    print(f"CPU rownolegle ({cpu_count()}): najlepsza trasa = {best_par:.2f}, czas = {t_par:.2f} s")

    t = time.perf_counter()
    best_gpu = ga_gpu(kernels)
    t_gpu = time.perf_counter() - t
    print(f"GPU (CUDA):       najlepsza trasa = {best_gpu:.2f}, czas = {t_gpu:.2f} s")

    print(f"Przyspieszenie CPU rownolegle vs 1 watek: {t_seq / t_par:.2f}x")
    print(f"Przyspieszenie GPU vs 1 watek:            {t_seq / t_gpu:.2f}x")

CPU 1 watek:      najlepsza trasa = 2377.16, czas = 12.93 s
CPU rownolegle (2): najlepsza trasa = 2377.16, czas = 12.89 s
GPU (CUDA):       najlepsza trasa = 2372.06, czas = 0.18 s
Przyspieszenie CPU rownolegle vs 1 watek: 1.00x
Przyspieszenie GPU vs 1 watek:            72.18x
